# Fish Speech — Клонирование голоса с картавостью (русский язык)

**Почему Fish Speech:**
- Активно поддерживается (2024-2025)
- Нативная поддержка русского языка
- Fine-tuning сохраняет особенности произношения
- Работает на Python 3.12 без конфликтов

**Порядок:**
1. Шаг 1 — проверь GPU (T4)
2. Шаг 2 — установка (~3 мин)
3. Шаг 3 — загрузи аудио (3-10 минут голоса)
4. Шаг 4 — нарезка и расшифровка
5. Шаг 5 — fine-tuning
6. Шаг 6 — генерация и скачивание

## Шаг 1: Проверь GPU
Если нет — **Runtime → Change runtime type → T4 GPU**

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
if result.returncode == 0:
    print(f'GPU найден: {result.stdout.strip()}')
else:
    print('GPU НЕ найден! Runtime → Change runtime type → T4 GPU')

## Шаг 2: Установка
После завершения — **Runtime → Restart session**, затем продолжай с Шага 3.

In [ ]:
# Клонируем Fish Speech
!git clone https://github.com/fishaudio/fish-speech.git
%cd fish-speech

# Устанавливаем зависимости
!pip install -q -e .
!pip install -q openai-whisper pydub
!apt-get install -q -y ffmpeg

print()
print('=' * 50)
print('Установка завершена!')
print('Сделай: Runtime → Restart session')
print('После перезапуска — Шаг 3')
print('=' * 50)

## Шаг 3: Загрузка модели и аудио
После Restart session начинай отсюда.

In [ ]:
import os, torch
%cd /content/fish-speech

# Скачиваем веса модели
!huggingface-cli download fishaudio/fish-speech-1.5 --local-dir checkpoints/fish-speech-1.5

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('Модель загружена!')

In [ ]:
from google.colab import files
import os

os.makedirs('/content/my_voice/wavs', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

print('Загрузи аудиофайл (WAV или MP3, минимум 3 минуты для fine-tuning)...')
uploaded = files.upload()

for filename in uploaded.keys():
    audio_path = f'/content/{filename}'
    with open(audio_path, 'wb') as f:
        f.write(uploaded[filename])
    print(f'Загружен: {filename} ({len(uploaded[filename])/1024/1024:.1f} MB)')

print(f'\nПуть: {audio_path}')

## Шаг 4: Нарезка и авто-расшифровка

In [ ]:
from pydub import AudioSegment
from pydub.silence import split_on_silence
import whisper, torch, os

print('Загружаем аудио...')
audio = AudioSegment.from_file(audio_path)
audio = audio.set_channels(1).set_frame_rate(22050)
print(f'Длина: {len(audio)/1000:.1f} сек')

# Нарезаем по тишине
chunks = split_on_silence(audio, min_silence_len=400,
                          silence_thresh=audio.dBFS - 16, keep_silence=100)
good_chunks = [c for c in chunks if 2000 <= len(c) <= 10000]

if len(good_chunks) < 10:
    print(f'Мало фрагментов ({len(good_chunks)}), нарезаем по 5 сек...')
    good_chunks = [audio[i:i+5000] for i in range(0, len(audio)-5000, 5000)]

# Сохраняем
chunk_paths = []
for i, chunk in enumerate(good_chunks):
    path = f'/content/my_voice/wavs/chunk_{i:04d}.wav'
    chunk.export(path, format='wav')
    chunk_paths.append(path)
print(f'Сохранено {len(chunk_paths)} фрагментов')

# Расшифровка через Whisper
print('\nРасшифровываем через Whisper...')
wmodel = whisper.load_model('medium')
transcriptions = []

for i, path in enumerate(chunk_paths):
    result = wmodel.transcribe(path, language='ru', fp16=torch.cuda.is_available())
    text = result['text'].strip()
    if len(text) > 5:
        transcriptions.append((path, text))
    if i % 10 == 0:
        print(f'[{i+1}/{len(chunk_paths)}] {text[:50]}')

print(f'\nГотово: {len(transcriptions)} фрагментов с транскрипцией')

In [ ]:
# Создаём датасет в формате Fish Speech
# Каждый аудиофайл + рядом .lab файл с текстом
for wav_path, text in transcriptions:
    lab_path = wav_path.replace('.wav', '.lab')
    with open(lab_path, 'w', encoding='utf-8') as f:
        f.write(text)

print(f'Датасет создан: {len(transcriptions)} пар аудио+текст')
print('Пример:')
for wav_path, text in transcriptions[:3]:
    print(f'  {os.path.basename(wav_path)}: {text[:60]}')

## Шаг 5: Fine-tuning
Займёт 20-60 минут. Можно оставить работать.

In [ ]:
%cd /content/fish-speech

# Создаём конфиг для fine-tuning
import json

# Список файлов датасета
filelist = [wav for wav, _ in transcriptions]
filelist_path = '/content/my_voice/filelist.txt'
with open(filelist_path, 'w') as f:
    for path in filelist:
        f.write(path + '\n')

print(f'Файлов для обучения: {len(filelist)}')

# Запускаем fine-tuning
!python fish_speech/train.py \
    --config-name firefly_gan_vq \
    train_dataset.filelist={filelist_path} \
    val_dataset.filelist={filelist_path} \
    trainer.max_steps=1000 \
    trainer.val_check_interval=200 \
    +trainer.accelerator=gpu \
    +trainer.devices=1 \
    output_dir=/content/fish_finetuned 2>&1 | tail -50

print('Fine-tuning завершён!')

## Шаг 6: Генерация голоса

In [ ]:
import IPython.display as ipd

%cd /content/fish-speech

# Эталонное аудио — первый фрагмент из датасета
ref_audio = transcriptions[0][0]
ref_text  = transcriptions[0][1]

# ==============================
# ВВЕДИ СВОЙ ТЕКСТ ЗДЕСЬ
# ==============================
TEXT = "Привет! Это клон моего голоса с сохранением картавости. Рыжая лиса прыгнула через реку."

out_path = '/content/output/result.wav'

# Шаг 1: кодируем эталонный голос
!python tools/vqgan/inference.py \
    -i "{ref_audio}" \
    --checkpoint-path checkpoints/fish-speech-1.5/firefly-gan-vq-fsq-8x1024-21hz-generator.pth \
    -o /content/output/ref_codes.npy

# Шаг 2: генерируем токены
!python tools/llama/generate.py \
    --text "{TEXT}" \
    --prompt-text "{ref_text}" \
    --prompt-tokens /content/output/ref_codes.npy \
    --checkpoint-path checkpoints/fish-speech-1.5 \
    --num-samples 1 \
    --output-dir /content/output/tokens

# Шаг 3: декодируем в аудио
!python tools/vqgan/inference.py \
    -i /content/output/tokens/codes_0.npy \
    --checkpoint-path checkpoints/fish-speech-1.5/firefly-gan-vq-fsq-8x1024-21hz-generator.pth \
    -o {out_path}

print('Результат:')
ipd.display(ipd.Audio(out_path))

In [ ]:
from google.colab import files
files.download(out_path)
print('Скачано!')